In [ ]:
import os
import torch

# This tells the computer: "Pretend there are ZERO GPUs available"
os.environ["CUDA_VISIBLE_DEVICES"] = ""

# This tells PyTorch: "Don't even try to check for a GPU"
torch.cuda.is_available = lambda : False
torch.device = lambda x: torch.device('cpu')

In [1]:
from langchain_classic.retrievers.ensemble import EnsembleRetriever

c:\Users\Aaru\OneDrive\Desktop\Informal Multilingual RAG\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [3]:
DATA_DIR = "./data"
PERSIST_DIR = "./mumbai_worker_db"
EMBEDDING_MODEL = "BAAI/bge-m3"

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cuda'}  
)
vectorstore = Chroma(persist_directory=PERSIST_DIR, embedding_function=embeddings)
dense_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 7})

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 417.04it/s, Materializing param=pooler.dense.weight]                               


In [5]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
loader = PyPDFDirectoryLoader(DATA_DIR)
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, separators=["\n\n", "\n", "।", ".", " ", ""])
chunks = splitter.split_documents(docs)

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 7

In [6]:

ensemble_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, bm25_retriever],
    weights=[0.65, 0.35]
)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "" # MUST be first
import torch
torch.cuda.is_available = lambda : False

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline

model_id = "google/gemma-2-2b-it"


model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="cpu",        
    torch_dtype=torch.float32, 
    low_cpu_mem_usage=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.7
)

llm = HuggingFacePipeline(pipeline=pipe)

Loading weights: 100%|██████████| 288/288 [04:30<00:00,  1.07it/s, Materializing param=model.norm.weight]                                
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [8]:
prompt = PromptTemplate.from_template("""You are a helpful assistant for gig and informal workers in Mumbai.
Use ONLY the retrieved information.
Answer in simple language, step-by-step, in the same language as the question.
Be kind and encouraging.

Retrieved context:
{context}

Question: {question}

Answer:
- Direct advice from documents
- Next steps if applicable
- ALWAYS end with this disclaimer:
  "यह केवल सामान्य जानकारी है। यह कानूनी सलाह नहीं है। कृपया किसी NGO, श्रम कार्यालय या सरकारी अधिकारी से सलाह लें। स्रोत: {sources}"

Sources: {sources}""")



In [9]:

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def format_sources(docs):
    
    return ", ".join(set(os.path.basename(doc.metadata.get("source", "Unknown")) for doc in docs))


rag_chain = (
    {
        "context": ensemble_retriever | format_docs,
        "question": RunnablePassthrough(),
        "sources": ensemble_retriever | format_sources  
    }
    | prompt 
    | llm 
    | StrOutputParser()
)